In [5]:
# For automatic reloading of project modules
%load_ext autoreload
%autoreload 2

# Allow imports from the parent project folder
import sys
sys.path.append('..')

import numpy as np
import pandas as pd

import stoch_models
import numerical_schemes
import option_contracts
import analytical_results
import greeks

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [6]:
# Model parameters
model = stoch_models.GBM(r=0.05, sigma=0.2)

# Market and contract parameters
S = 100.0
t = 0.0
T = 1.0
K = 100.0
tau = T - t

option = option_contracts.EuropeanCall(
    strike=K,
    maturity=T,
)

# Simulation parameters
N = 100_000
M = 1
dx = 0.001
scheme = numerical_schemes.exact_step

times = np.linspace(t, T, M + 1)

# Common random numbers for all perturbed valuations
rng = np.random.default_rng(42)
Z = rng.standard_normal(size=(N, M))

In [7]:
result_delta = greeks.central_diff_delta(
    model=model,
    option_contract=option,
    scheme=scheme,
    S=S,
    N=N,
    times=times,
    dx=dx,
    normal_draws=Z,
)

result_vega = greeks.central_diff_vega(
    model=model,
    option_contract=option,
    scheme=scheme,
    S=S,
    N=N,
    times=times,
    dx=dx,
    normal_draws=Z,
)

result_gamma = greeks.central_diff_gamma(
    model=model,
    option_contract=option,
    scheme=scheme,
    S=S,
    N=N,
    times=times,
    dx=dx,
    normal_draws=Z,
)

In [8]:
bs_delta = analytical_results.black_scholes_delta(
    model=model, option_contract=option, S=S, t=t
)

bs_vega = analytical_results.black_scholes_vega(
    model=model, option_contract=option, S=S, t=t
)

bs_gamma = analytical_results.black_scholes_gamma(
    model=model, option_contract=option, S=S, t=t
)

In [9]:
results = [
    (result_delta, bs_delta),
    (result_vega, bs_vega),
    (result_gamma, bs_gamma),
]

comparison = pd.DataFrame([
    {
        "Greek": result.greek_name.capitalize(),
        "Analytical": benchmark,
        "Monte Carlo": result.estimate,
        "Error": result.estimate - benchmark,
        "Standard Error": result.standard_error,
        "CI Lower": result.ci_lower,
        "CI Upper": result.ci_upper,
        "Bump Size": result.bump_size,
    }
    for result, benchmark in results
])

comparison.set_index("Greek").round(6)

,Analytical,Monte Carlo,Error,Standard Error,CI Lower,CI Upper,Bump Size
Greek,,,,,,,
Delta,0.636831,0.634534,-0.002297,0.001823,0.630961,0.638107,0.1000
Vega,37.524035,37.480075,-0.043960,0.241220,37.007283,37.952867,0.0002
Gamma,0.018762,0.018182,-0.000580,0.001075,0.016075,0.020289,0.1000
